<a href="https://colab.research.google.com/github/silvia-dev-prog/ai-agents-for-beginners/blob/main/C%C3%B3pia_de_RL_Aula2_Sincrona_DDPG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from scipy.integrate import solve_ivp
from stable_baselines3 import DDPG
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv
import os
import matplotlib.pyplot as plt




ModuleNotFoundError: No module named 'stable_baselines3'

In [ ]:
# Ambiente Gym Customizado
class CSTRControlEnv(gym.Env):
    def __init__(self):
        super().__init__()

        # Espaços de ação contínuos
        self.action_space = spaces.Box(low=np.array([0.0, 0.0]), high=np.array([ 1,  1]), dtype=np.float32)

        # Espaços de observação: Erros normalizados de temperatura e pressão
        self.observation_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)

        # Parâmetros do reator
        self.params = {
            "V": 1.0,  # Volume do reator (m³)
            "Cp": 4.18e3,  # Capacidade térmica do líquido (J/kg·K)
            "rho": 1000,  # Densidade do líquido (kg/m³)
            "k_reaction": 5e-1,  # Constante de reação exotérmica
            "delta_Hr": -5e5,  # Calor liberado pela reação (J/mol)
            "P_inert": 1e5,  # Pressão inicial do gás inerte (Pa)
            "T_cool": 298.0,  # Temperatura do agente de resfriamento (K)
            "T_ambient": 298.0,  # Temperatura ambiente (K)
            "tau": 100.0,  # Constante de troca térmica (s)
            "T_setpoint": 350.0,  # Setpoint de temperatura (K)
            "P_setpoint": 2e5,  # Setpoint de pressão (Pa)
            "R": 8.314,  # Constante universal dos gases (J/mol·K)
        }

        # Vazões manipuladas
        self.coolant_flow = 0.0  # Vazão inicial do agente de resfriamento (kg/s)
        self.vent_flow = 0.0  # Vazão inicial do gás liberado (kg/s)

        self.reset()

    def reset(self, seed = None):
        # Condições iniciais do reator
        self.state = {
            "T": 298.0,  # Temperatura inicial (K)
            "P": 101325,  # Pressão inicial (Pa)
        }
        self.iteration = 0
        return self._get_normalized_state()

    def step(self, action):
        # Aplicar ações diretamente como vazões
        self.coolant_flow = 2*np.clip(action[0], 0, 1)
        self.vent_flow = 0.01*np.clip(action[1], 0, 1)

        # Resolver as EDOs do reator
        t_span = [0, 10]  # Passo de tempo de 10 segundos
        y0 = [self.state["T"], self.state["P"]]  # Condições iniciais
        sol = solve_ivp(self._reactor_model, t_span, y0, args=(self.coolant_flow, self.vent_flow))

        # Atualizar o estado do reator
        self.state["T"], self.state["P"] = sol.y[:, -1]

        # Calcular erros e recompensa
        error_T = (self.state["T"] - self.params["T_setpoint"]) / (350 - 300)
        error_P = (self.state["P"] - self.params["P_setpoint"]) / (4e5 - 0.5e5)
        reward = 2- (abs(error_T) + abs(error_P))  # Recompensa baseada nos erros absolutos

        # Incrementar o contador de iterações
        self.iteration += 1
         # Verificar condições de término
        truncated = self.iteration >= 300
        terminated = (
            self.state["T"] < 273 or self.state["T"] > 450 or
            self.state["P"] < 0.5e5 or self.state["P"] > 5e5
        )

        return self._get_normalized_state(), 10*reward, terminated, truncated, {}

    def _reactor_model(self, t, y, coolant_flow, vent_flow):
        T, P = y
        params = self.params

        # Balanço de energia
        Q_reaction = -params["k_reaction"] * params["delta_Hr"]
        Q_cooling = coolant_flow * params["Cp"] * (T - params["T_cool"])
        dT_dt = (Q_reaction - Q_cooling) / (params["rho"] * params["Cp"] * params["V"])

        # Balanço de pressão
        gas_generation = params["k_reaction"]  # Geração de gás pela reação (mol/s)
        dP_dt = (
            gas_generation * params["R"] * T / params["V"] -
            vent_flow * P / params["V"]
        )

        return [dT_dt, dP_dt]

    def _get_normalized_state(self):
        # Normalizar os erros de temperatura e pressão
        error_T = (self.state["T"] - self.params["T_setpoint"]) / (400 - 273)
        error_P = (self.state["P"] - self.params["P_setpoint"]) / (3e5 - 0.5e5)
        return np.array([error_T, error_P], dtype=np.float32)
    def render(self, mode='ansi'):
        print(self.iteration, self.coolant_flow, self.vent_flow,self.state)


In [ ]:
# Função para salvar e reavaliar o melhor modelo
def evaluate_best_model(model, env):
    obs = env.reset()
    T_values, P_values, flow0, flow1 = [], [], [],  []
    i=0

    while i<300:
        i+=1
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, trunc, term, _ = env.step(action)
        T_values.append(env.state["T"])
        P_values.append(env.state["P"])
        flow0.append(env.coolant_flow)
        flow1.append(env.vent_flow)

    plt.figure(figsize=(6,3))
    plt.plot(T_values, label="Temperatura (K)")
    plt.axhline(env.params["T_setpoint"], color="red", linestyle="--", label="Setpoint de Temperatura")
    plt.xlabel("Passos de Tempo")
    plt.ylabel("Valores")
    plt.legend()

    plt.figure(figsize=(6, 3))
    plt.plot(P_values, label="Pressão (Pa)", color="orange")
    plt.axhline(env.params["P_setpoint"], color="green", linestyle="--", label="Setpoint de Pressão")
    plt.xlabel("Passos de Tempo")
    plt.ylabel("Valores")
    plt.legend()

    plt.figure(figsize=(6, 3))
    plt.plot(flow0, label="Coolant Flow ", color="orange")
    plt.xlabel("Passos de Tempo")
    plt.ylabel("Valores")
    plt.legend()

    plt.figure(figsize=(6, 3))
    plt.plot(flow1, label="Vent Flow", color="orange")
    plt.xlabel("Passos de Tempo")
    plt.ylabel("Valores")
    plt.legend()

    plt.grid()
    plt.show()



In [ ]:
# Configuração do ambiente
env = DummyVecEnv([lambda: CSTRControlEnv()])

# Hiperparâmetros do DDPG
hyperparams = {'gamma': 0.95, 'learning_rate': 0.00001, 'buffer_size': 100000, 'batch_size': 512, 'train_freq': (2, 'episode'), 'gradient_steps': 1, 'policy_kwargs': {'net_arch': [64, 64]}}


# Inicializar o modelo
model = DDPG("MlpPolicy", env, **hyperparams, verbose=0)

# Treinar o modelo com callback para salvar o melhor
log_dir = "./2logs095-03/"
os.makedirs(log_dir, exist_ok=True)
eval_callback = EvalCallback(env, best_model_save_path=log_dir,
                             log_path=log_dir, eval_freq=35000,
                             deterministic=True, render=False)

model.learn(total_timesteps=700000, callback=eval_callback)


In [ ]:

# Reavaliar o melhor modelo encontrado
best_model = DDPG.load(f"{log_dir}/best_model.zip")
evaluate_best_model(best_model, env.envs[0])